In [1]:
import requests
from sqlite3 import connect
import pandas as pd

conn = connect("../StocksPlatform/app.db")

In [50]:
symbols = pd.read_sql_query("SELECT Name, Symbol, Country FROM Assets", conn)
symbols = symbols[(symbols["Country"] == "Norway")]
symbols

,Name,Symbol,Country
41,Genetic Analysis AS,GEAN,Norway
71,GULF KEYSTONE PETROLEUM LTD,GKP,Norway
112,CONSTELLATION OIL SERVICES HOLDING S.A.,COSH,Norway
129,REFUELS N.V.,REFL,Norway
162,SPAREBANK 1 NORDMØRE,SNOR,Norway
...,...,...,...
11956,CYVIZ AS,CYVIZ,Norway
12079,ELLIPTIC LABORATORIES ASA,ELABS,Norway
12156,TEKNA HOLDING ASA,TEKNA,Norway
12200,STOLT-NIELSEN,SNI,Norway


In [51]:
symbols[symbols["Symbol"] == "EQNR"]

,Name,Symbol,Country
6045,EQUINOR,EQNR,Norway


In [55]:
symbols_concat = ""
# responses = []
base_url = "https://api.e24.no/bors/tags?symbols=" # EQNR.OSE,YAR.OSE,AAPL.NAS
for i, row in symbols.iterrows():
    if row["Country"] == "United States":
        symbols_concat += f"{row['Symbol']}.NAS,"
    elif row["Country"] == "Norway":
        symbols_concat += f"{row['Symbol']}.OSE,"
    if i % 9 == 9 - 1:
        print("Fetching E24 tag IDs for symbols: " + symbols_concat[:-1])
        response = requests.get(base_url + symbols_concat[:-1])
        response_json = response.json()
        symbols_concat = ""
        responses.append(response_json)

Fetching E24 tag IDs for symbols: GEAN.OSE,GKP.OSE
Fetching E24 tag IDs for symbols: COSH.OSE,REFL.OSE,SNOR.OSE,MAS.OSE,BSP.OSE,ABL.OSE,ITERA.OSE,ASA.OSE,ABTEC.OSE,EPR.OSE,ISLAX.OSE,MOWI.OSE,PCIB.OSE,DSRT.OSE
Fetching E24 tag IDs for symbols: APR.OSE,PNOR.OSE,EQVA.OSE,PLGC.OSE
Fetching E24 tag IDs for symbols: INDCT.OSE
Fetching E24 tag IDs for symbols: INSTA.OSE,BRUT.OSE,ENERG.OSE
Fetching E24 tag IDs for symbols: PLSV.OSE
Fetching E24 tag IDs for symbols: BORR.OSE,TRSB.OSE,NHY.OSE,KCC.OSE,KOMPL.OSE,BEWI.OSE,NKR.OSE,HYPRO.OSE,BRG.OSE,GSF.OSE
Fetching E24 tag IDs for symbols: 2020.OSE,MORG.OSE,ININ.OSE
Fetching E24 tag IDs for symbols: NISB.OSE
Fetching E24 tag IDs for symbols: CRNA.OSE,BWLPG.OSE,PLT.OSE,KRAB.OSE
Fetching E24 tag IDs for symbols: DOFG.OSE,RING.OSE,OTEC.OSE,SMCRT.OSE
Fetching E24 tag IDs for symbols: ARCH.OSE,HELG.OSE,NORBT.OSE
Fetching E24 tag IDs for symbols: STECH.OSE,NRC.OSE,TECH.OSE,NTG.OSE,OTOVO.OSE,SCATC.OSE
Fetching E24 tag IDs for symbols: AKOBO.OSE
Fetching E2

In [56]:
print("Fetching E24 tag IDs for symbols: " + symbols_concat[:-1])
response = requests.get(base_url + symbols_concat[:-1])
response_json = response.json()
symbols_concat = ""
responses.append(response_json)

Fetching E24 tag IDs for symbols: SOAG.OSE,HAV.OSE,STB.OSE,GEM.OSE,MEDI.OSE,RIVER.OSE,CYVIZ.OSE,ELABS.OSE,TEKNA.OSE,SNI.OSE,NOKUSD=X.OSE


In [60]:
symbol_e24tag_mapping = {}
for r in responses:
    for item in r["items"]:
        symbol = None
        for metadata in item["model"]["metadata"]:
            if "value" not in metadata: continue
            if "." in metadata["value"] and "www" not in metadata["value"]:
                symbol = metadata["value"]
                break
        if symbol:
            symbol = symbol.replace(".NAS", "").replace(".OSE", "")
            symbol_e24tag_mapping[symbol] = item["id"]
symbol_e24tag_mapping

{'SNOR': '7f6142bf-a28c-48ae-8740-17d5d1e718cd',
 'ITERA': '3e89fd54-7222-41e1-9dbb-bc005191bee3',
 'ABL': '4bb70596-3bd9-4c42-a46b-964b2c651091',
 'EPR': '6d9318cd-2a47-4b9f-bb42-330235f02754',
 'ASA': 'ba126a9e-2432-4023-8192-510283a447e8',
 'EQVA': '000825cd-ac0d-45f6-b8d9-ccfd1f70768c',
 'MOWI': 'd84adba1-ea4b-4b8b-953a-2a1dd09a1f0c',
 'LPG': '768bf4a2-a12c-43b3-a4d5-90991f0c1c94',
 'APR': '974fa226-62ed-4373-8871-75f62e221809',
 'PCIB': 'df1e6295-c179-40c6-8978-9d493591dcf5',
 'PNOR': 'fc696279-8d37-4296-845a-f6d51b9c3e17',
 'NHY': '59d15ea1-f991-4c05-b083-1032ab81168d',
 'KCC': '9afa0619-8ede-4ab1-b707-17aa6e34d5d0',
 'BORR': 'db3de1bb-2ca2-4a2a-86c0-c2a0400c095e',
 'HYPRO': '1d2243ed-dad2-48fc-86a5-533d391d6cb2',
 'NKR': 'a5eedeeb-b2dc-41ff-ad60-1848ed6f50c2',
 'KOMPL': '8860098b-d5e8-4171-b6c5-91b20bc75333',
 'BEWI': 'df8ac0a7-d3e6-4ddf-85f1-6e36ef6e270f',
 'MORG': '20541b88-031d-4b3e-a9fa-504a9ac18126',
 'GSF': 'c9eb25bf-90b6-4cdb-9457-4a74e9a6d0a4',
 '2020': '6609db43-beb8-4c

In [61]:
symbol_e24tag_mapping["EQNR"]

'5ee6e30c-8658-476a-a65e-dd8f4da594ae'

In [66]:
nordnet_csv = pd.read_csv("../StocksPlatform/Services/Seeding/Data/nordnet_all_stocks.csv")
nordnet_csv["E24Tag"] = nordnet_csv["Symbol"].map(symbol_e24tag_mapping)
nordnet_csv.to_csv("../StocksPlatform/Services/Seeding/Data/nordnet_all_stocks.csv", index=False)